# K-Fold Cross Validation — LoRA vs Full Fine-Tuning
**STAT 453, Spring 2026 | University of Wisconsin-Madison**

---

## What is this notebook?

This notebook compares two fine-tuned models on the RECAST-30K dataset:
- **Rinkle's model** — Llama 3.2 1B with LoRA adapters (rank=8, ~0.5% of parameters trained)
- **Mark's model** — Llama 3.2 1B fully fine-tuned (100% of parameters trained)

Both models are given the **same test instructions** and we check how well each one follows the constraints in those instructions.

---

## What is K-Fold Cross Validation?

Instead of testing on just one chunk of data, K-Fold splits the full dataset into K equal pieces (folds) and tests on each piece separately. The final score is the average across all folds.

**Why this matters:** A single test split might accidentally contain easy or hard examples. K-Fold tests on ALL examples equally, giving a fairer, more reliable score.

```
RECAST-30K (30,000 examples) split into 5 folds of 6,000 each:

Round 1: [TEST Fold 1] [skip 2] [skip 3] [skip 4] [skip 5] → score
Round 2: [skip 1] [TEST Fold 2] [skip 3] [skip 4] [skip 5] → score
Round 3: [skip 1] [skip 2] [TEST Fold 3] [skip 4] [skip 5] → score
Round 4: [skip 1] [skip 2] [skip 3] [TEST Fold 4] [skip 5] → score
Round 5: [skip 1] [skip 2] [skip 3] [skip 4] [TEST Fold 5] → score

Final Score = average of all 5 round scores
```

**Speed note:** Testing all 6,000 examples per fold would take too long (~50 hrs). We sample 200 examples from each fold → 1,000 total → ~1.5 hours on T4 GPU.

---

## What is CSR (Constraint Satisfaction Rate)?

CSR measures how well a model follows the constraints in an instruction. It is a number between **0 and 1** — higher is better.

Example: An instruction has 4 constraints. The model satisfies 3 of them → **CSR = 3/4 = 0.75**

---

## How to run this notebook

1. Connect to **T4 or A100 GPU** (Runtime → Change runtime type → GPU)
2. Set `HF_TOKEN` in Colab Secrets (🔑 left sidebar)
3. Update the **3 paths** in Step 3
4. Run **every cell top to bottom** — do NOT skip any cell

| Step | What it does | Approx time |
|------|-------------|-------------|
| 1 | Install libraries + login | 2 min |
| 2 | Import modules | <1 min |
| 3 | Set configuration | <1 min |
| 4 | Load dataset + create folds | 2 min |
| 5 | Define constraint validators | <1 min |
| 6 | Define generation functions | <1 min |
| 7 | **Run K-Fold evaluation** | ~1.5 hrs |
| 8 | Compute scores | <1 min |
| 9 | Plot results | <1 min |
| 10 | Print final summary | <1 min |

## Step 1 — Install Libraries and Login

**What this does:** Installs all Python packages needed and logs into HuggingFace.

**Before running:** Add `HF_TOKEN` to Colab Secrets (🔑 icon on left sidebar). Get your token from `https://huggingface.co/settings/tokens`.

**Expected output:** `Logged in to HuggingFace.`

In [ ]:
!pip install -q transformers peft datasets accelerate matplotlib pandas scikit-learn

In [ ]:
import os
from huggingface_hub import login

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN').strip()
except Exception:
    hf_token = os.environ.get('HF_TOKEN', '').strip()

login(token=hf_token)
print("Logged in to HuggingFace.")

## Step 2 — Import Python Modules

**What each package does:**
- `torch` — runs models on GPU
- `peft` — loads Rinkle's LoRA adapter on top of base model
- `transformers` — loads both models and tokenizer
- `sklearn KFold` — splits dataset into K equal folds automatically
- `pandas` — builds the comparison table
- `matplotlib` — draws the charts
- `gc` — clears GPU memory between models

**Expected output:** `GPU: Tesla T4` or `GPU: A100`. If you see `GPU available: False` — change runtime to GPU first.

In [ ]:
import gc, json, os, re, random
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from peft import PeftModel
from sklearn.model_selection import KFold
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU. Go to Runtime → Change runtime type → GPU")

## Step 3 — Configuration

**This is the only cell you need to change before running.**

| Variable | Who provides it | What to put |
|----------|----------------|-------------|
| `LORA_ADAPTER_PATH` | Rinkle | Path to `lora_adapter/` folder |
| `FULL_FT_MODEL_PATH` | Mark | Path to `finetuned/` folder |
| `DATASET_PATH` | Shreyas | Path to `recast_30k_clean.jsonl` |

**Expected output:** Prints a configuration summary.

In [ ]:
# ── UPDATE THESE THREE PATHS ──────────────────────────────────────────────────
BASE_MODEL         = "meta-llama/Llama-3.2-1B-Instruct"
LORA_ADAPTER_PATH  = "/content/outputs/lora_r8_0.0001/lora_adapter"  # Rinkle's adapter
FULL_FT_MODEL_PATH = "/content/output/finetuned"                      # Mark's full FT model
DATASET_PATH       = "/content/recast_30k_clean.jsonl"               # Shreyas's dataset

# ── K-FOLD SETTINGS (leave as-is) ────────────────────────────────────────────
K_FOLDS          = 5    # 5 rounds of testing
SAMPLES_PER_FOLD = 200  # examples per round (200x5 = 1,000 total)
RANDOM_SEED      = 42   # fixed seed = reproducible results

# ── OUTPUT FOLDER ─────────────────────────────────────────────────────────────
RESULTS_DIR = "/content/kfold_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Configuration set:")
print(f"  LoRA adapter:     {LORA_ADAPTER_PATH}")
print(f"  Full FT model:    {FULL_FT_MODEL_PATH}")
print(f"  Dataset:          {DATASET_PATH}")
print(f"  K folds:          {K_FOLDS}")
print(f"  Samples per fold: {SAMPLES_PER_FOLD}")
print(f"  Total evaluated:  {K_FOLDS * SAMPLES_PER_FOLD} examples")

## Step 4 — Load Dataset and Create K Folds

**What this does:**
1. Reads the full RECAST-30K jsonl file line by line
2. Extracts the instruction and response from each row
3. Shuffles with a fixed seed (reproducible)
4. Divides into 5 equal chunks using sklearn KFold
5. Samples 200 examples from each chunk

**Expected output:** Shows the size of each fold.

In [ ]:
def load_all_examples(path):
    """
    Reads RECAST jsonl and returns list of examples.
    Handles both field name conventions (Rinkle's and Mark's scripts).
    """
    examples = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            instruction = row.get("winner_prompt", row.get("input", ""))
            response    = row.get("response_of_winner_prompt",
                          row.get("winner_response", row.get("output", "")))
            if instruction and response:
                examples.append({
                    "instruction":   instruction,
                    "gold_response": response,
                    "raw":           row,  # full row kept for validators
                })
    return examples


print(f"Loading dataset...")
all_examples = load_all_examples(DATASET_PATH)
print(f"Total examples: {len(all_examples):,}")

# Shuffle with fixed seed for reproducibility
random.seed(RANDOM_SEED)
random.shuffle(all_examples)

# Create 5 equal folds
kf      = KFold(n_splits=K_FOLDS, shuffle=False)
indices = list(range(len(all_examples)))
folds   = []

for fold_num, (_, test_idx) in enumerate(kf.split(indices)):
    sampled_idx   = random.sample(list(test_idx), min(SAMPLES_PER_FOLD, len(test_idx)))
    fold_examples = [all_examples[i] for i in sampled_idx]
    folds.append(fold_examples)
    print(f"  Fold {fold_num+1}: {len(test_idx):,} total → {len(fold_examples)} sampled")

print(f"\nTotal to evaluate: {sum(len(f) for f in folds)} examples")

## Step 5 — Constraint Validators

**What this does:** Defines one function per constraint type. Each function takes the model's response and the raw RECAST row, and returns `True` (constraint satisfied) or `False` (not satisfied).

| Validator | What it checks |
|-----------|---------------|
| `check_length_words` | Word count is in the required range |
| `check_length_sentences` | Sentence count is under the limit |
| `check_keyword` | Required word appears N times |
| `check_start_with` | First word matches requirement |
| `check_end_with` | Last word matches requirement |
| `check_format` | Uses required format like `<<title>>` |
| `check_tone` | No informal words (proxy for formal tone) |
| Style / Role / Numerical | Pass-through (too complex to automate) |

**Expected output:** `Validators ready for 9 constraint categories.`

In [ ]:
def check_length_words(response, raw):
    """Word count must be within the range specified in the constraint."""
    try:
        constraints = raw.get("added_constraint", {}).get("Length", [])
        wc = len(response.split())
        for c in constraints:
            nums = re.findall(r'\d+', c)
            if len(nums) >= 2 and "word" in c.lower():
                if not (int(nums[0]) <= wc <= int(nums[1])):
                    return False
        return True
    except Exception:
        return True

def check_length_sentences(response, raw):
    """Sentence count must be under the limit specified in the constraint."""
    try:
        constraints = raw.get("added_constraint", {}).get("Length", [])
        sc = len(re.split(r'[.!?]+', response.strip()))
        for c in constraints:
            nums = re.findall(r'\d+', c)
            if nums and "sentence" in c.lower():
                if sc > int(nums[0]):
                    return False
        return True
    except Exception:
        return True

def check_keyword(response, raw):
    """Required keywords must appear the required number of times."""
    try:
        constraints = raw.get("added_constraint", {}).get("Keyword", [])
        rl = response.lower()
        for c in constraints:
            m = re.search(r'["\u201c\u201d]([^"]+)["\u201c\u201d].*?(\d+)\s*times', c, re.IGNORECASE)
            if m and rl.count(m.group(1).lower()) < int(m.group(2)):
                return False
        return True
    except Exception:
        return True

def check_start_with(response, raw):
    """Response must begin with the required word."""
    try:
        constraints = raw.get("added_constraint", {}).get(
            "Strat_With", raw.get("added_constraint", {}).get("Start_With", [])
        )
        for c in constraints:
            m = re.search(r'["\u201c\u201d]([^"]+)["\u201c\u201d]', c)
            if m:
                words = response.strip().lower().split()
                if not words or words[0] != m.group(1).strip().lower():
                    return False
        return True
    except Exception:
        return True

def check_end_with(response, raw):
    """Response must end with the required word."""
    try:
        constraints = raw.get("added_constraint", {}).get("End_With", [])
        for c in constraints:
            m = re.search(r'["\u201c\u201d]([^"]+)["\u201c\u201d]', c)
            if m:
                words = re.findall(r'\w+', response.lower())
                if not words or words[-1] != m.group(1).strip().lower():
                    return False
        return True
    except Exception:
        return True

def check_format(response, raw):
    """Response must use required format markers like <<title>>."""
    try:
        constraints = raw.get("added_constraint", {}).get("Format", [])
        for c in constraints:
            if "<<" in c and not re.search(r'<<.+?>>', response):
                return False
        return True
    except Exception:
        return True

def check_tone(response, raw):
    """Response must not contain informal language (proxy for formal tone)."""
    informal = ["gonna", "wanna", "gotta", "kinda", "dunno", "lol", "omg"]
    return not any(w in response.lower() for w in informal)

# These are too complex to check automatically — always return True
def check_style(response, raw):        return True
def check_role_playing(response, raw): return True
def check_numerical(response, raw):    return True

# Map each category to its validator functions
VALIDATORS = {
    "Length":                [check_length_words, check_length_sentences],
    "Keyword":               [check_keyword],
    "Start_With":            [check_start_with],
    "End_With":              [check_end_with],
    "Format":                [check_format],
    "Tone":                  [check_tone],
    "Style":                 [check_style],
    "Role_Playing":          [check_role_playing],
    "Numerical_Constraints": [check_numerical],
}

print(f"Validators ready for {len(VALIDATORS)} constraint categories.")

## Step 6 — Generation and Scoring Functions

**What this does:** Defines the functions that:
1. Feed an instruction to a model and get its response
2. Score that response against all constraint validators → CSR value
3. Loop through all examples in one fold

**Key detail:** We use `do_sample=False` (greedy decoding) — the model always picks the most likely next word. Same input always gives the same output, which makes the comparison fair and reproducible.

**Expected output:** `Generation functions ready.`

In [ ]:
# Llama 3's exact chat format — must match what was used during training
PROMPT_TEMPLATE = (
    "<|begin_of_text|>"
    "<|start_header_id|>user<|end_header_id|>\n\n"
    "{instruction}"
    "<|eot_id|>"
    "<|start_header_id|>assistant<|end_header_id|>\n\n"
)


def generate_response(model, tokenizer, instruction, max_new_tokens=512):
    """
    Feeds the instruction into the model and returns the generated response.
    Uses greedy decoding (do_sample=False) so results are reproducible.
    """
    prompt = PROMPT_TEMPLATE.format(instruction=instruction)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():  # saves GPU memory — no gradients needed for evaluation
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # greedy: always pick most likely next token
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    # Decode only the new tokens (skip the input prompt)
    return tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()


def score_response(response, raw):
    """
    Runs all constraint validators on the model's response.
    Returns per-category True/False and overall CSR (0.0 to 1.0).
    CSR = fraction of constraint categories satisfied.
    """
    results, all_passed = {}, []
    for cat, fns in VALIDATORS.items():
        passed = all(fn(response, raw) for fn in fns)  # all validators must pass
        results[cat] = passed
        all_passed.append(passed)
    results["csr"] = sum(all_passed) / len(all_passed)
    return results


def evaluate_fold(model, tokenizer, fold_examples, model_name, fold_num):
    """
    Runs the model on every example in one fold.
    Prints progress every 50 examples so you can see it working.
    Returns a list of scored results.
    """
    results = []
    model.eval()  # evaluation mode: disables dropout
    for i, ex in enumerate(fold_examples):
        if i % 50 == 0:
            print(f"    [{model_name}] Fold {fold_num} — {i}/{len(fold_examples)} done...")
        response = generate_response(model, tokenizer, ex["instruction"])
        scores   = score_response(response, ex["raw"])
        results.append({"fold": fold_num, "model": model_name, **scores})
    print(f"    [{model_name}] Fold {fold_num} complete.")
    return results


print("Generation functions ready.")

## Step 7 — Run K-Fold Evaluation

**This is the main loop — approximately 1.5 hours on T4 GPU.**

**What happens in each of the 5 folds:**
```
1. Load base model + LoRA adapter (Rinkle's model)
2. Run 200 examples through it, score each response
3. Delete LoRA model from GPU memory
4. Load Mark's full fine-tuned model
5. Run the SAME 200 examples through it, score each response
6. Delete Full FT model from GPU memory
7. Save this fold's results to disk (checkpoint)
8. Move to next fold
```

**Why we delete models between each:** T4 GPU has 16GB VRAM. Each model uses ~4GB. We cannot keep both loaded at once safely, so we alternate.

**Checkpoints:** Results are saved after every fold. If Colab crashes, the completed folds are not lost.

**Watch for this progress pattern:**
```
=======================================================
  FOLD 1 / 5 — 200 examples
=======================================================
  Loading LoRA model...
  [LoRA] Fold 1 — 0/200 done...
  [LoRA] Fold 1 — 50/200 done...
  LoRA done. GPU freed.
  Loading Full FT model...
  [Full_FT] Fold 1 — 0/200 done...
  ...
  Fold 1 saved.
```

In [ ]:
all_results = []  # collects every scored example from every fold, both models

for fold_num, fold_examples in enumerate(folds, start=1):
    print(f"\n{'='*55}")
    print(f"  FOLD {fold_num} / {K_FOLDS} — {len(fold_examples)} examples")
    print(f"{'='*55}")

    # Load tokenizer once — converts text to token IDs, shared by both models
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # ── RINKLE'S LORA MODEL ──────────────────────────────────────────────────
    print("  Loading LoRA model (frozen base + trained adapter)...")
    base       = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, torch_dtype=torch.bfloat16, device_map="auto")
    lora_model = PeftModel.from_pretrained(base, LORA_ADAPTER_PATH)

    lora_fold_results = evaluate_fold(
        lora_model, tokenizer, fold_examples, "LoRA", fold_num)
    all_results.extend(lora_fold_results)

    # Free GPU memory before loading next model
    del lora_model, base
    gc.collect()
    torch.cuda.empty_cache()
    print("  LoRA done. GPU memory freed.")

    # ── MARK'S FULL FT MODEL ─────────────────────────────────────────────────
    print("  Loading Full FT model (all weights updated)...")
    full_model = AutoModelForCausalLM.from_pretrained(
        FULL_FT_MODEL_PATH, torch_dtype=torch.bfloat16, device_map="auto")

    full_fold_results = evaluate_fold(
        full_model, tokenizer, fold_examples, "Full_FT", fold_num)
    all_results.extend(full_fold_results)

    del full_model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print("  Full FT done. GPU memory freed.")

    # Save checkpoint after each fold
    with open(f"{RESULTS_DIR}/results_fold_{fold_num}.json", "w") as f:
        json.dump(lora_fold_results + full_fold_results, f, indent=2)
    print(f"  Fold {fold_num} saved to {RESULTS_DIR}/results_fold_{fold_num}.json")

# Save all results combined
with open(f"{RESULTS_DIR}/all_results.json", "w") as f:
    json.dump(all_results, f, indent=2)

print(f"\n{'='*55}")
print(f"All {K_FOLDS} folds complete! Total scored: {len(all_results)}")
print(f"{'='*55}")

## Step 8 — Compute Final K-Fold Scores

**What this does:**
1. Groups results by model and fold
2. Computes average CSR per fold for each model
3. Averages those per-fold scores → **final score**
4. Computes standard deviation → shows how consistent each model is

**What the ± means:** Small ± = model is consistent across folds (good). Large ± = performance varies a lot by example set.

**Expected output:** Per-fold CSR table + full comparison table with all constraint categories.

In [ ]:
df   = pd.DataFrame(all_results)
cats = list(VALIDATORS.keys())


def fold_summary(model_name):
    """
    Computes per-fold CSR then averages across folds.
    This is the correct K-Fold methodology — each fold gets equal weight.
    """
    model_df    = df[df["model"] == model_name]
    fold_scores = []
    for fold_num in range(1, K_FOLDS + 1):
        fold_df = model_df[model_df["fold"] == fold_num]
        if len(fold_df) == 0:
            continue
        fold_score = {"fold": fold_num, "CSR": fold_df["csr"].mean()}
        for cat in cats:
            if cat in fold_df.columns:
                fold_score[cat] = fold_df[cat].mean()
        fold_scores.append(fold_score)

    fold_df_all = pd.DataFrame(fold_scores)
    summary = {}
    for m in ["CSR"] + cats:
        if m in fold_df_all.columns:
            summary[m] = {"mean": fold_df_all[m].mean(), "std": fold_df_all[m].std()}
    return summary, fold_df_all


lora_summary,    lora_fold_df    = fold_summary("LoRA")
full_ft_summary, full_ft_fold_df = fold_summary("Full_FT")

print("LoRA — CSR per fold:")
print(lora_fold_df[["fold", "CSR"]].to_string(index=False))
print("\nFull FT — CSR per fold:")
print(full_ft_fold_df[["fold", "CSR"]].to_string(index=False))

# Build comparison table
rows = []
for m in ["CSR"] + cats:
    if m in lora_summary and m in full_ft_summary:
        lm = lora_summary[m]["mean"] * 100
        ls = lora_summary[m]["std"]  * 100
        fm = full_ft_summary[m]["mean"] * 100
        fs = full_ft_summary[m]["std"]  * 100
        winner = "LoRA" if lm > fm else ("Full_FT" if fm > lm else "Tie")
        rows.append({"Metric": m, "LoRA mean±std": f"{lm:.1f}±{ls:.1f}",
                     "Full FT mean±std": f"{fm:.1f}±{fs:.1f}", "Winner": winner})

comparison = pd.DataFrame(rows)
print("\n" + "="*65)
print("  K-Fold Results — mean±std across 5 folds (%)")
print("="*65)
print(comparison.to_string(index=False))
print("="*65)
comparison.to_csv(f"{RESULTS_DIR}/kfold_comparison.csv", index=False)
print(f"\nSaved: {RESULTS_DIR}/kfold_comparison.csv")

## Step 9 — Plot Results

**What you get — 3 charts:**

1. **CSR per Fold (line chart)** — consistency check. Stable lines = reliable model.
2. **Per-Category CSR (bar chart)** — which constraint types each model handles best.
3. **Final Overall Score (bar chart)** — the headline result for your paper.

**Expected output:** Charts displayed inline + saved as PNG.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fold_nums = list(range(1, K_FOLDS + 1))

# Chart 1: Per-fold CSR line — shows consistency across data slices
axes[0].plot(fold_nums, lora_fold_df["CSR"].values*100,
             marker="o", label="LoRA (Rinkle)", color="#2196F3", linewidth=2)
axes[0].plot(fold_nums, full_ft_fold_df["CSR"].values*100,
             marker="s", label="Full FT (Mark)", color="#FF5722", linewidth=2)
axes[0].set_xlabel("Fold Number")
axes[0].set_ylabel("CSR (%)")
axes[0].set_title("CSR per Fold\n(consistency check)")
axes[0].set_xticks(fold_nums)
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].set_ylim(0, 110)

# Chart 2: Per-category bar chart with error bars
cat_ml = [lora_summary.get(c,{}).get("mean",0)*100 for c in cats]
cat_sl = [lora_summary.get(c,{}).get("std",0)*100  for c in cats]
cat_mf = [full_ft_summary.get(c,{}).get("mean",0)*100 for c in cats]
cat_sf = [full_ft_summary.get(c,{}).get("std",0)*100  for c in cats]
x, w   = range(len(cats)), 0.35
axes[1].bar([i-w/2 for i in x], cat_ml, w, yerr=cat_sl, capsize=4,
            label="LoRA (Rinkle)", color="#2196F3", alpha=0.85)
axes[1].bar([i+w/2 for i in x], cat_mf, w, yerr=cat_sf, capsize=4,
            label="Full FT (Mark)", color="#FF5722", alpha=0.85)
axes[1].set_xticks(list(x))
axes[1].set_xticklabels([c.replace("_","\n") for c in cats], fontsize=7)
axes[1].set_ylabel("Satisfaction Rate (%)")
axes[1].set_title("Per-Category CSR\n(mean±std across 5 folds)")
axes[1].legend()
axes[1].set_ylim(0, 120)
axes[1].grid(axis="y", alpha=0.3)

# Chart 3: Final overall score — main result for paper
lf = lora_summary.get("CSR",{}).get("mean",0)*100
ff = full_ft_summary.get("CSR",{}).get("mean",0)*100
le = lora_summary.get("CSR",{}).get("std",0)*100
fe = full_ft_summary.get("CSR",{}).get("std",0)*100
bars = axes[2].bar([0,1],[lf,ff],yerr=[le,fe],capsize=8,
                   color=["#2196F3","#FF5722"],alpha=0.85,width=0.5)
axes[2].set_xticks([0,1])
axes[2].set_xticklabels(["LoRA\n(Rinkle)","Full FT\n(Mark)"],fontsize=11)
axes[2].set_ylabel("CSR (%)")
axes[2].set_title("Final K-Fold CSR Score\n(use this in your paper)")
axes[2].set_ylim(0,120)
axes[2].grid(axis="y",alpha=0.3)
for bar, val in zip(bars,[lf,ff]):
    axes[2].text(bar.get_x()+bar.get_width()/2, val+2,
                 f"{val:.1f}%",ha="center",va="bottom",fontsize=12,fontweight="bold")

plt.suptitle(
    f"K={K_FOLDS} Fold Cross Validation — LoRA vs Full Fine-Tuning\n"
    "Llama 3.2 1B Instruct on RECAST-30K | STAT 453, Spring 2026",
    fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/kfold_comparison_plot.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Plot saved: {RESULTS_DIR}/kfold_comparison_plot.png")

## Step 10 — Final Score Summary

**This is what goes in your paper.**

You will see:
- **One overall CSR score per model** (0 to 1) — the headline number
- **Which model wins per constraint category** — the detailed breakdown
- **± std** — how reliable each score is

Higher CSR = better constraint following.

In [ ]:
lora_csr = lora_summary["CSR"]["mean"]
lora_std = lora_summary["CSR"]["std"]
full_csr = full_ft_summary["CSR"]["mean"]
full_std = full_ft_summary["CSR"]["std"]

lora_wins = sum(1 for r in comparison.to_dict("records") if r["Winner"] == "LoRA")
full_wins = sum(1 for r in comparison.to_dict("records") if r["Winner"] == "Full_FT")
ties      = sum(1 for r in comparison.to_dict("records") if r["Winner"] == "Tie")

print("\n" + "="*55)
print("  FINAL SCORE  (put this in your paper)")
print("="*55)
print(f"  LoRA (Rinkle)  →  CSR = {lora_csr:.4f}  (±{lora_std:.4f})")
print(f"  Full FT (Mark) →  CSR = {full_csr:.4f}  (±{full_std:.4f})")
print("")
winner = "LoRA" if lora_csr > full_csr else "Full FT"
print(f"  Winner: {winner}  (difference = {abs(lora_csr-full_csr):.4f})")
print("="*55)

print("\n  Per-Category Breakdown:")
print(f"  {'Category':<25} {'LoRA':>8} {'Full FT':>10} {'Winner':>12}")
print("  " + "-"*57)
for m in comparison.to_dict("records"):
    cat = m["Metric"]
    if cat == "CSR":
        continue
    lv  = lora_summary.get(cat, {}).get("mean", 0)
    fv  = full_ft_summary.get(cat, {}).get("mean", 0)
    win = "LoRA ✓" if lv > fv else ("Full FT ✓" if fv > lv else "Tie")
    print(f"  {cat:<25} {lv:>8.4f} {fv:>10.4f} {win:>12}")

print("\n" + "="*55)
print(f"  Category wins → LoRA: {lora_wins} | Full FT: {full_wins} | Ties: {ties}")
print("="*55)
print(f"\nFiles saved to {RESULTS_DIR}/")
print("  kfold_comparison.csv       ← table for paper")
print("  kfold_comparison_plot.png  ← figure for paper")
print("  all_results.json           ← raw data")
print("  results_fold_1..5.json     ← per-fold checkpoints")